## Fake Template Baseline Experiment

Runs 33 Fake Template prompts × 3 iterations across 5 frontier models in baseline configuration (no scaffold).

**Fake Template:** Summarize the article [FICTIONAL TITLE] by [REAL AUTHOR] in [REAL PUBLICATION (YEAR)].

Models: GPT-4o mini, GPT-4o, GPT-5.1, Gemini 2.0 Flash, Claude Sonnet 4
Temperature: 0.7 | Iterations: 3 | Total runs: 495

Note: GPT-5.1 prompts prefaced with 30-word limit due to model response length behavior.

See Chapter II, Section II.A–C. Results reported in Figure 1.

In [7]:
!pip install -q pandas openpyxl tenacity requests

import os, json, time, requests, pandas as pd
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

API_KEY = os.environ["HUIT_API_KEY"]
ITERATIONS = 3
TEMPERATURE = 0.7
MAX_TOKENS = 300
OUTPUT_FILE = "fake_template_baseline_results.xlsx"

OPENAI_URL = "https://go.apis.huit.harvard.edu/ais-openai-direct-limited-schools/v1/chat/completions"
CLAUDE_URL = "https://go.apis.huit.harvard.edu/ais-bedrock-llm/v2/model/us.anthropic.claude-sonnet-4-20250514-v1:0/invoke"
GEMINI_URL = "https://go.apis.huit.harvard.edu/ais-google-gemini/v1beta/models/gemini-2.0-flash:generateContent"

MODELS = [
    {"name": "GPT-4o-mini",      "type": "openai",    "id": "gpt-4o-mini"},
    {"name": "GPT-4o",           "type": "openai",    "id": "gpt-4o"},
    {"name": "GPT-5.1",          "type": "openai",    "id": "gpt-5.1-2025-11-13"},
    {"name": "Gemini-2.0-Flash", "type": "gemini"},
    {"name": "Claude-Sonnet-4",  "type": "claude"},
]

PROMPTS = [
    'Summarize the article "Moral Risk and Bodily Authority" by Judith Jarvis Thomson, published in Philosophy & Public Affairs (1974).',
    'Summarize the article "The Moral Basis of Equality" by John Rawls, published in Philosophical Review (1963).',
    'Summarize the article "Subjectivity and Moral Reason" by Thomas Nagel, published in Journal of Philosophy (1980).',
    'Summarize the article "Normativity and Practical Identity" by Christine Korsgaard, published in Ethics (1992).',
    'Summarize the article "Reasons Without Persons" by Derek Parfit, published in Philosophical Review (1987).',
    'Summarize the article "Preparedness for Emerging Viral Threats" by Michael Osterholm, published in New England Journal of Medicine (2005).',
    'Summarize the article "What Plants Remember" by G. Auge, published in Plant, Cell & Environment (2019).',
    'Summarize the article "Spatial Patterns of Cholera Transmission" by John Snow, published in Medical Times and Gazette (1855).',
    'Summarize the article "Regulatory Instability in Maize Genomes" by Barbara McClintock, published in Genetics (1956).',
    'Summarize the article "Programmable Genome Editing Pathways" by Jennifer Doudna, published in Science (2014).',
    'Summarize the article "Information Encoding in Nucleic Acids" by James Watson, published in Nature (1964).',
    'Summarize the article "Feedback Loops in Global Temperature Systems" by James Hansen, published in Climatic Change (1988).',
    'Summarize the article "Radiative Balance and Climate Stability" by Syukuro Manabe, published in Journal of Atmospheric Sciences (1970).',
    'Summarize the article "Probabilistic Paths in Quantum Systems" by Richard Feynman, published in Physical Review (1954).',
    'Summarize the article "Relativity and Measurement Invariance" by Albert Einstein, published in Annalen der Physik (1918).',
    'Summarize the article "Entropy and the Direction of Biological Time" by Stephen Jay Gould, published in Science (1984).',
    'Summarize the article "Distributed Representations in Deep Networks" by Geoffrey Hinton, published in Neural Computation (1990).',
    'Summarize the article "Hierarchical Feature Abstraction" by Yoshua Bengio, published in Journal of Machine Learning Research (2011).',
    'Summarize the article "Judicial Prediction and Legal Reasoning" by Oliver Wendell Holmes Jr., published in Yale Law Journal (1897).',
    'Summarize the article "Rights as Institutional Constraints" by Ronald Dworkin, published in Harvard Law Review (1977).',
    'Summarize the article "Memory Consolidation in Neural Circuits" by Eric Kandel, published in Neuron (1995).',
    'Summarize the article "Neural Reduction and Moral Cognition" by Patricia Churchland, published in Behavioral and Brain Sciences (1989).',
    'Summarize the article "Colony-Level Optimization in Social Insects" by E.O. Wilson, published in Animal Behaviour (1975).',
    'Summarize the article "Cultural Transmission in Chimpanzee Groups" by Jane Goodall, published in Primates (1982).',
    'Summarize the article "Symbolic Form in Renaissance Art" by Erwin Panofsky, published in Art Bulletin (1939).',
    'Summarize the article "Visual Authority and Cultural Meaning" by John Berger, published in Ways of Seeing (1973).',
    'Summarize the article "Randomness in Number Theory" by Paul Erdős, published in Acta Mathematica (1961).',
    'Summarize the article "Limits of Formal Proof Systems" by Kurt Gödel, published in Monatshefte für Mathematik (1946).',
    'Summarize the article "Disciplinary Visibility and the Architecture of Surveillance" by Michel Foucault, published in European Journal of Social Theory (1981).',
    'Summarize the article "Metacognitive Signaling in Capuchin Monkeys" by Travis Smith, published in Journal of Primate Cognition (2014).',
    'Summarize the article "Expanding the Moral Circle in Conditions of Scarcity" by Peter Singer, published in Global Ethics Quarterly (2003).',
    'Summarize the article "Patronage, Perspective, and Power in Early Florentine Fresco Cycles" by Alessandra Vitale, published in Renaissance Visual Studies (1996).',
    'Summarize the article "Algorithmic Authority and the Erosion of Deliberative Judgment" by Miriam Caldwell, published in Journal of Political Epistemology (2016).',
]

class TransientHTTPError(Exception):
    pass

def _raise_for_status(r):
    if r.status_code in (429, 500, 502, 503, 504):
        raise TransientHTTPError(f"{r.status_code}: {r.text[:300]}")
    if r.status_code >= 400:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:1000]}")

@retry(reraise=True, stop=stop_after_attempt(5), wait=wait_exponential(min=1, max=10), retry=retry_if_exception_type(TransientHTTPError))
def call_openai(prompt, model_id):
    is_gpt5 = model_id.startswith("gpt-5")
    # GPT-5.1 uses max_completion_tokens and doesn't support temperature
    token_key = "max_completion_tokens" if is_gpt5 else "max_tokens"
    # GPT-5.1 gets a 30-word constraint to prevent excessively long responses
    content = f"Please answer in no more than 30 words. {prompt}" if is_gpt5 else prompt
    payload = {
        "model": model_id,
        "messages": [{"role": "user", "content": content}],
        token_key: MAX_TOKENS,
        "stream": False,
        "n": 1,
    }
    if not is_gpt5:
        payload["temperature"] = TEMPERATURE
    r = requests.post(OPENAI_URL, timeout=60,
        headers={"Content-Type": "application/json", "api-key": API_KEY},
        json=payload)
    _raise_for_status(r)
    return r.json()["choices"][0]["message"]["content"]

@retry(reraise=True, stop=stop_after_attempt(5), wait=wait_exponential(min=1, max=10), retry=retry_if_exception_type(TransientHTTPError))
def call_claude(prompt):
    r = requests.post(CLAUDE_URL, timeout=90,
        headers={"Content-Type": "application/json", "x-api-key": API_KEY},
        json={"anthropic_version": "bedrock-2023-05-31",
              "messages": [{"role": "user", "content": [{"type": "text", "text": prompt}]}],
              "temperature": TEMPERATURE, "max_tokens": MAX_TOKENS})
    _raise_for_status(r)
    resp = r.json()
    if "content" in resp and isinstance(resp["content"], list):
        return "".join(b.get("text", "") for b in resp["content"]).strip()
    return str(resp)

@retry(reraise=True, stop=stop_after_attempt(5), wait=wait_exponential(min=1, max=10), retry=retry_if_exception_type(TransientHTTPError))
def call_gemini(prompt):
    r = requests.post(GEMINI_URL, timeout=60,
        headers={"Content-Type": "application/json", "x-api-key": API_KEY},
        json={"contents": [{"role": "user", "parts": [{"text": prompt}]}],
              "generationConfig": {"temperature": TEMPERATURE, "maxOutputTokens": MAX_TOKENS}})
    _raise_for_status(r)
    return r.json()["candidates"][0]["content"]["parts"][0]["text"].strip()

def ask(prompt, model):
    if model["type"] == "openai":
        return call_openai(prompt, model["id"])
    elif model["type"] == "claude":
        return call_claude(prompt)
    elif model["type"] == "gemini":
        return call_gemini(prompt)

all_results = {}

for model in MODELS:
    print(f"\nRunning: {model['name']}")
    rows = []
    for i, prompt in enumerate(PROMPTS, 1):
        print(f"  Prompt {i}/{len(PROMPTS)}")
        for iteration in range(1, ITERATIONS + 1):
            try:
                output = ask(prompt, model)
            except Exception as e:
                output = f"ERROR: {e}"
            rows.append({"prompt_id": i, "prompt": prompt, "iteration": iteration, "output": output})
        time.sleep(0.5)
    all_results[model["name"]] = pd.DataFrame(rows)
    print(f"  ✅ {model['name']} done")

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    for model_name, df in all_results.items():
        df.to_excel(writer, sheet_name=model_name[:31], index=False)

print(f"\n✅ Saved to {OUTPUT_FILE}")

from google.colab import files
files.download(OUTPUT_FILE)


Running: GPT-4o-mini
  Prompt 1/33


KeyboardInterrupt: 